# SO(3) benchmarking and profiling

Runs the generic benchmark helper with an explicit SO(3) backend, then profiles the same benchmark call with the standard-library `cProfile` profiler.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
try:
    candidates.extend(path for path in start.iterdir() if path.is_dir())
except OSError:
    pass

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "pyproject.toml").exists()
        and (path / "src" / "equivariant_polynomials").is_dir()
        and (path / "benchmarks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find the project root. Start Jupyter from the project directory "
        "or from its notebooks directory."
    )

for import_path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    import_path = str(import_path)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\micha\OneDrive\Desktop\Oden\Notes\Equivariant Polynomial Tensor Functions\Python\equivariant_polynomials


In [2]:
import cProfile
import io
import pstats

from pprint import pp

from benchmarks import benchmark
from equivariant_polynomials.groups.so3 import SO3RepresentationTheory, hilbert_series_so3, hilbert_series_so3_multigraded


In [3]:
random_seed = 12345
input_irreps = (2, 3)
input_multiplicities = (2, 1)
output_irreps = (4,)
max_degree = 9
invariant_irrep = 0
modulus = 2521
profile_sort = "cumtime"
profile_rows = 25

benchmark_config = {
    "hilbert_series": hilbert_series_so3,
    "hilbert_series_multigraded": hilbert_series_so3_multigraded,
    "input_irreps": input_irreps,
    "input_multiplicities": input_multiplicities,
    "max_degree": max_degree,
    "invariant_irrep": invariant_irrep,
    "random_seed": random_seed,
    "modulus": modulus,
}


## Benchmark


In [4]:
theory = SO3RepresentationTheory()
summaries = []

for output_irrep in output_irreps:
    print(f"\nRun: output_irrep={output_irrep}, max_degree={max_degree}")
    summaries.append(
        benchmark(
            theory=theory,
            output_irrep=output_irrep,
            **benchmark_config,
        )
    )

summaries = tuple(summaries)
pp(summaries)



Run: output_irrep=4, max_degree=9
algebra generators: 5.896 s
module generators: 47.073 s
total: 52.988 s
({'scenario': {'input_irreps': (2, 3),
               'input_multiplicities': (2, 1),
               'output_irrep': 4,
               'invariant_irrep': 0,
               'max_degree': 9,
               'random_seed': 12345,
               'modulus': 2521},
  'algebra': {'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739),
              'generator_counts': (0, 0, 4, 7, 14, 26, 52, 68, 60, 39),
              'matches_known_generators': True,
              'generator_seconds': 5.89625530000194},
  'module': {'hilbert_dimensions': (0,
                                    0,
                                    6,
                                    21,
                                    87,
                                    273,
                                    807,
                                    2109,
                                    5205,
                  

## cProfile


In [5]:
profiled_output_irrep = output_irreps[0]
profiler = cProfile.Profile()
profile_summary = profiler.runcall(
    benchmark,
    theory=SO3RepresentationTheory(),
    output_irrep=profiled_output_irrep,
    **benchmark_config,
)

stats_stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stats_stream)
stats.strip_dirs().sort_stats(profile_sort).print_stats(profile_rows)
print(stats_stream.getvalue())

profile_summary


algebra generators: 8.690 s
module generators: 70.297 s
total: 79.042 s
         41676350 function calls (41664735 primitive calls) in 79.005 seconds

   Ordered by: cumulative time
   List reduced from 389 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   79.042   79.042 benchmark.py:58(benchmark)
        2    0.009    0.005   79.041   39.521 benchmark.py:74(run_generators)
        2    0.015    0.008   78.893   39.447 generators.py:100(extract_independent_generators)
      248    0.016    0.000   74.188    0.299 generators.py:420(_stream_content_generators)
      952    0.005    0.000   72.844    0.077 generators.py:543(_syndrome_batches)
     4003    0.010    0.000   69.798    0.017 generators.py:43(__iter__)
    10635    0.006    0.000   69.781    0.007 generators.py:50(<genexpr>)
      153    0.004    0.000   69.775    0.456 bases.py:278(schur_functor_basis)
   492202    0.469    0.000   66.642    0

{'scenario': {'input_irreps': (2, 3),
  'input_multiplicities': (2, 1),
  'output_irrep': 4,
  'invariant_irrep': 0,
  'max_degree': 9,
  'random_seed': 12345,
  'modulus': 2521},
 'algebra': {'hilbert_dimensions': (1, 0, 4, 7, 24, 54, 156, 340, 817, 1739),
  'generator_counts': (0, 0, 4, 7, 14, 26, 52, 68, 60, 39),
  'matches_known_generators': True,
  'generator_seconds': 8.689969399973052},
 'module': {'hilbert_dimensions': (0,
   0,
   6,
   21,
   87,
   273,
   807,
   2109,
   5205,
   11919),
  'generator_counts': (0, 0, 6, 21, 63, 147, 264, 284, 133, 35),
  'matches_known_generators': True,
  'generator_seconds': 70.29716700001154},
 'total_seconds': 79.04160920000868}